# Replicating Tessler et al. (2024) Fig. 4C — minority weight in Habermas Machine group statements

Pipeline: Sentence-T5 embeddings → per-question position axis (negating → affirming) → position component scores →
convex regression of group-statement scores on constituent opinion scores, per level of division → minority weight
(sum of minority coefficients), averaged over levels.

Paper's numbers to compare against (main text, RQ3): average minority size 29%; initial statements: minority weight 0.29;
revised statements: 0.36 (SE 0.03, t = 2.64). Fig. 4A: r = 0.64 between opinion position score and pre-deliberation rating.
Fig. 4B: 96% of group-statement scores within the range of the group's opinions.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from hm_fig4c import pipeline as P
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

EMB_DIR = os.environ.get("HM_EMB_DIR", "../embeddings/st5-base")   # embedding cache to use
AXIS_METHOD = os.environ.get("HM_AXIS_METHOD", "affine")             # 'affine' (0 = negating, 1 = affirming) or 'unit'
N_BOOT = int(os.environ.get("HM_N_BOOT", "500"))
MODEL_TAG = os.path.basename(EMB_DIR.rstrip("/"))
OUT_DIR = f"../results/{MODEL_TAG}"; os.makedirs(OUT_DIR, exist_ok=True)
print(EMB_DIR, AXIS_METHOD, N_BOOT)

## 1. Score all texts on the position axis

In [ ]:
opinions, statements, questions = P.score_all("../prepared", EMB_DIR, method=AXIS_METHOD)
cov = pd.DataFrame({"opinions_scored": opinions.groupby("cohort")["score"].apply(lambda s: s.notna().mean()),
                    "initial_scored": statements.groupby("cohort")["initial_score"].apply(lambda s: s.notna().mean()),
                    "revised_scored": statements.groupby("cohort")["revised_score"].apply(lambda s: s.notna().mean()),
                    "n_rounds": statements.groupby("cohort").size()})
cov.round(3)

## 2. Fig. 4A check — opinion position score vs pre-deliberation position rating (paper: r = 0.64)

In [ ]:
rows = {c: P.fig4a_correlation(P.select_cohort(opinions, c)) for c in ["cohort1", "cohort2", "cohort3", "cohorts_1_3", "cohort4", "training", "vca"]}
fig4a = pd.DataFrame(rows).T; fig4a

In [ ]:
d = P.select_cohort(opinions, "cohorts_1_3").dropna(subset=["score", "pre_rating"])
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(d["pre_rating"] + np.random.uniform(-.15, .15, len(d)), d["score"], s=4, alpha=.25)
means = d.groupby("pre_rating")["score"].mean()
ax.plot(means.index, means.values, "o-", color="k", ms=5)
ax.set_xlabel("Pre-deliberation position rating (1 = strongly disagree, 7 = strongly agree)"); ax.set_ylabel("Position component score")
ax.set_title(f"Cohorts 1-3: r = {fig4a.loc['cohorts_1_3','r']:.2f}  (paper: 0.64)", fontsize=9); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig4a.png", dpi=150)

## 3. Fig. 4B check — statement scores relative to the group's opinions (paper: 96% within range)

In [ ]:
fig4b = {c: P.fig4b_within_range(P.select_cohort(opinions, c), P.select_cohort(statements, c)) for c in ["cohorts_1_3", "training", "vca"]}
pd.DataFrame({(c, s): v for c, dd in fig4b.items() for s, v in dd.items()}).T

In [ ]:
d_op = P.select_cohort(opinions, "cohorts_1_3"); d_st = P.select_cohort(statements, "cohorts_1_3")
fig, ax = plt.subplots(figsize=(5, 3.2))
for vals, lab, col in [(d_op["score"], "opinions", "tab:red"), (d_st["initial_score"], "initial statements", "tab:blue"), (d_st["revised_score"], "revised statements", "tab:purple")]:
    ax.hist(vals.dropna(), bins=60, density=True, histtype="step", lw=1.5, label=lab, color=col)
ax.set_xlabel("Position component score (0 = negating, 1 = affirming)"); ax.legend(frameon=False, fontsize=8); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig4b.png", dpi=150)

## 4. Fig. 4C — minority weight via convex regression (primary specification)

Cohorts 1–3 pooled; minority = smaller side of neutral on the pre-deliberation rating; neutral raters dropped
(this reproduces the paper's 29% average minority size); rounds with a tie or no dissent excluded; columns ordered as in the data.

In [ ]:
res = P.run_minority_analysis(opinions, statements, cohort="cohorts_1_3", neutral="drop_participant", order="data", n_boot=N_BOOT)
P.save_results(res, f"{OUT_DIR}/fig4c_primary.json")
summary = P.summarize(res); summary

In [ ]:
P.per_level_table(res)

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 3.6))
P.plot_fig4c(res, ax=ax, title=f"Cohorts 1-3, {MODEL_TAG} (n = {res['initial']['n_rounds']} rounds)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig4c.png", dpi=200)

## 5. Sensitivity analyses
Each row varies one choice relative to the primary specification.

In [ ]:
variants = {
  "primary (cohorts 1-3, drop neutral participant, data order)": dict(cohort="cohorts_1_3"),
  "neutral: drop whole group": dict(cohort="cohorts_1_3", neutral="drop_group"),
  "neutral: count as non-minority": dict(cohort="cohorts_1_3", neutral="as_majority"),
  "column order: sorted by score": dict(cohort="cohorts_1_3", order="sorted"),
  "column order: random": dict(cohort="cohorts_1_3", order="random"),
  "cohort 1 only": dict(cohort="cohort1"), "cohort 2 only": dict(cohort="cohort2"), "cohort 3 only": dict(cohort="cohort3"),
  "cohort 4 (critique exclusion)": dict(cohort="cohort4"),
  "training data": dict(cohort="training"), "virtual citizens' assembly": dict(cohort="vca"),
  "initial stmt from fine-tuned generator only": dict(cohort="cohorts_1_3", statement_filter="initial_gen_api == 'hydra_70b_generative'"),
  "initial stmt from base Chinchilla generator only": dict(cohort="cohorts_1_3", statement_filter="initial_gen_api == 'chinchilla'"),
}
rows = []
for name, kw in variants.items():
    try:
        r = P.run_minority_analysis(opinions, statements, **kw)
        rows.append({"variant": name, "n_rounds": r["initial"]["n_rounds"], "true_share": r["initial"]["true_share"],
                     "initial_w": r["initial"]["weight"], "initial_se": r["initial"]["se"],
                     "revised_w": r["revised"]["weight"], "revised_se": r["revised"]["se"],
                     "revised_t_vs_true": r["revised"]["t_vs_true"], "diff": r["diff"]["weight"]})
    except Exception as e:
        rows.append({"variant": name, "error": str(e)[:80]})
sens = pd.DataFrame(rows); sens.to_csv(f"{OUT_DIR}/sensitivity.csv", index=False); sens.round(3)

## 6. Robustness: regress the full 768-d embedding (not just the position score) on convex combinations of opinion embeddings

In [ ]:
from hm_fig4c.analysis import assign_minority, convex_fit_with_se
from hm_fig4c.data import text_id
from hm_fig4c.embed import load_embeddings
lookup, mat = load_embeddings(EMB_DIR)
op = assign_minority(P.select_cohort(opinions, "cohorts_1_3"), neutral="drop_participant")
st = P.select_cohort(statements, "cohorts_1_3").set_index(P.KEY)
def vec_design(stmt_col):
    lv = {}
    for k, g in op.groupby(P.KEY):
        if k not in st.index or not isinstance(st.at[k, stmt_col], str): continue
        ids = [text_id(t) for t in g["opinion_text"]]; sid = text_id(st.at[k, stmt_col])
        if sid not in lookup or any(i not in lookup for i in ids): continue
        mn = g["is_minority"].values
        X = np.stack([mat[lookup[i]] for i in ids], axis=1)  # 768 x n
        X = np.concatenate([X[:, mn], X[:, ~mn]], axis=1)
        y = mat[lookup[sid]]
        key = (int(g["n_div"].iloc[0]), int(g["k_min"].iloc[0]))
        lv.setdefault(key, {"X": [], "y": []}); lv[key]["X"].append(X); lv[key]["y"].append(y)
    return {k: {"X": np.concatenate(v["X"]), "y": np.concatenate(v["y"]), "n_rounds": len(v["y"])} for k, v in lv.items()}
vec_rows = []
for stage, col in [("initial", "initial_text"), ("revised", "revised_text")]:
    D = vec_design(col); tot = sum(d["n_rounds"] for d in D.values()); acc_w = 0; acc_true = 0
    for (n, k), d in sorted(D.items()):
        if d["n_rounds"] < 10: continue
        w, mw, se, _, r2 = convex_fit_with_se(d["X"], d["y"], np.array([1.]*k + [0.]*(n-k)))
        acc_w += d["n_rounds"]/tot*mw; acc_true += d["n_rounds"]/tot*k/n
        vec_rows.append({"stage": stage, "n": n, "k": k, "n_rounds": d["n_rounds"], "minority_weight": mw, "r2": r2})
    vec_rows.append({"stage": stage, "n": "all", "k": "-", "n_rounds": tot, "minority_weight": acc_w, "true_share": acc_true})
vec = pd.DataFrame(vec_rows); vec.to_csv(f"{OUT_DIR}/vector_regression.csv", index=False); vec.round(3)

## 7. Summary vs paper

In [ ]:
paper = {"minority_share": 0.29, "initial_w": 0.29, "revised_w": 0.36, "revised_se": 0.03, "fig4a_r": 0.64, "fig4b_within": 0.96}
ours = {"minority_share": res["initial"]["true_share"], "initial_w": res["initial"]["weight"], "initial_se": res["initial"]["se"],
        "revised_w": res["revised"]["weight"], "revised_se": res["revised"]["se"], "revised_t_vs_true": res["revised"]["t_vs_true"],
        "fig4a_r": fig4a.loc["cohorts_1_3", "r"], "fig4b_within_initial": fig4b["cohorts_1_3"]["initial_score"]["within"],
        "fig4b_within_revised": fig4b["cohorts_1_3"]["revised_score"]["within"], "n_rounds": res["initial"]["n_rounds"], "model": MODEL_TAG}
json.dump({"paper": paper, "ours": ours}, open(f"{OUT_DIR}/summary.json", "w"), indent=1, default=float)
pd.DataFrame({"paper": paper, "ours": ours}).round(3)